In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Input e output di Estimator

<Accordion>
<AccordionItem title="Versioni dei pacchetti">

Il codice in questa pagina è stato sviluppato con i seguenti requisiti.
Ti consigliamo di utilizzare queste versioni o versioni più recenti.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Questa pagina fornisce una panoramica degli input e degli output della primitiva Qiskit Runtime Estimator, che esegue carichi di lavoro su risorse di calcolo IBM Quantum&reg;. Estimator ti consente di definire in modo efficiente carichi di lavoro vettorizzati usando una struttura dati chiamata [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). Vengono usati come input per il metodo [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) della primitiva Estimator, che esegue il carico di lavoro definito come job. Poi, dopo che il job è stato completato, i risultati vengono restituiti in un formato che dipende sia dai PUB usati che dalle opzioni di runtime specificate dalla primitiva.
## Input
Ogni PUB ha questo formato:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

I `parameter values` opzionali possono essere un elenco o un singolo parametro. Gli elementi degli osservabili e dei valori dei parametri vengono combinati seguendo le regole di broadcasting di NumPy come descritto nell'argomento [Input e output delle primitive](primitive-input-output#broadcasting-rules), e viene restituita una stima del valore di aspettazione per ogni elemento della forma broadcast.

> **Note:** Se l'input contiene misurazioni, vengono ignorate.

Per la primitiva Estimator, un PUB può contenere al massimo quattro valori:
- Un singolo `QuantumCircuit`, che può contenere uno o più oggetti [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Un elenco di uno o più osservabili, che specificano i valori di aspettazione da stimare, disposti in un array (ad esempio, un singolo osservabile rappresentato come array 0-d, un elenco di osservabili come array 1-d e così via). I dati possono essere in uno qualsiasi dei formati `ObservablesArrayLike` come `Pauli`, `SparsePauliOp`, `PauliList` o `str`.
   > **Note:** - Gli osservabili che commutano **nello stesso PUB** vengono raggruppati usando [questo metodo](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Gli osservabili che commutano in PUB diversi, anche se hanno lo stesso Circuit, non vengono stimati usando la stessa misurazione. Ogni PUB rappresenta una base diversa per la misurazione e quindi sono necessarie misurazioni separate per ogni PUB.
>    - Per assicurarti che gli osservabili che commutano siano stimati usando la stessa misurazione, raggruppali nello stesso PUB.
- Una raccolta di valori dei parametri con cui associare il Circuit. Questo può essere specificato come un singolo oggetto array-like in cui l'ultimo indice è sugli oggetti `Parameter` del Circuit o omesso (o equivalentemente, impostato su `None`) se il Circuit non ha oggetti `Parameter`.
- (Opzionalmente) Una precisione target per i valori di aspettazione da stimare
---

Il seguente codice dimostra un esempio di set vettorizzati di input alla primitiva `Estimator` e li esegue su un backend IBM&reg; come singolo oggetto `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Output
Dopo che uno o più PUB vengono inviati a una QPU per l'esecuzione e un job viene completato con successo, i dati vengono restituiti come oggetto contenitore [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) accessibile chiamando il metodo `RuntimeJobV2.result()`.

Il `PrimitiveResult` contiene un elenco iterabile di oggetti [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) che contengono i risultati di esecuzione per ogni PUB.

Ogni elemento di questo elenco corrisponde a ogni PUB inviato al metodo `run()` della primitiva (ad esempio, un job inviato con 20 PUB restituirà un oggetto `PrimitiveResult` che contiene un elenco di 20 oggetti `PubResult`, uno corrispondente a ogni PUB).

Ogni `PubResult` per la primitiva Estimator contiene almeno un array di valori di aspettazione (`PubResult.data.evs`) e deviazioni standard associate (sia `PubResult.data.stds` che `PubResult.data.ensemble_standard_error` a seconda del `resilience_level` usato), ma può contenere più dati a seconda delle opzioni di mitigazione degli errori specificate.

Ogni oggetto `PubResult` possiede sia un attributo `data` che un attributo `metadata`.
    - L'attributo `data` è un [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) personalizzato che contiene i valori di misurazione effettivi, le deviazioni standard e così via.
    - Il `DataBin` ha vari attributi a seconda della forma o struttura del PUB associato e delle opzioni di mitigazione degli errori specificate dalla primitiva usata per inviare il job (ad esempio, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) o [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - L'attributo `metadata` contiene informazioni sulle opzioni di runtime e mitigazione degli errori usate (spiegate più avanti nella sezione [Metadati del risultato](#result-metadata) di questa pagina).

Di seguito è riportato uno schema visivo della struttura dati `PrimitiveResult` per l'output di Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

In breve, un singolo job restituisce un oggetto `PrimitiveResult` e contiene un elenco di uno o più oggetti `PubResult`. Questi oggetti `PubResult` memorizzano quindi i dati di misurazione per ogni PUB inviato al job.

Il seguente frammento di codice descrive il formato `PrimitiveResult` (e il relativo `PubResult`) per il job creato sopra.

In [2]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
